In [1]:
# =============================================================================
# 2026 제9회 지방선거 경합지역 당락 예측
# 모델: MLP + Platt Scaling + LR 앙상블
# 학습 데이터: 7회(2018), 8회(2022) 지선 실제 결과
# =============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. 7, 8회 지선 실제 결과 (선관위 공식 + 여론조사 격차)
#    poll_gap: 민주계 후보 - 국힘계 후보 여론조사 격차 (%p)
#    양수 = 민주계 우세 / 음수 = 국힘계 우세
# =============================================================================
historical_data = [
    # ── 8회 (2022)
    {'region': 'Seoul',     'year': 2022, 'poll_gap': -19.8, '민주당승': 0},  # 오세훈 당선
    {'region': 'Busan',     'year': 2022, 'poll_gap': -30.1, '민주당승': 0},  # 박형준 당선
    {'region': 'Daegu',     'year': 2022, 'poll_gap': -55.2, '민주당승': 0},  # 홍준표 당선
    {'region': 'Incheon',   'year': 2022, 'poll_gap':  -5.8, '민주당승': 0},  # 유정복 당선
    {'region': 'Gyeonggi',  'year': 2022, 'poll_gap':   1.2, '민주당승': 1},  # 김동연 당선 (0.16%p 초경합)
    {'region': 'Gangwon',   'year': 2022, 'poll_gap':  -9.1, '민주당승': 0},  # 김진태 당선
    {'region': 'Chungbuk',  'year': 2022, 'poll_gap': -12.3, '민주당승': 0},  # 김영환 당선
    {'region': 'Chungnam',  'year': 2022, 'poll_gap':  -5.9, '민주당승': 0},  # 김태흠 당선
    {'region': 'Jeonbuk',   'year': 2022, 'poll_gap':  18.4, '민주당승': 0},  # 김관영 당선 (국민의당)
    {'region': 'Gyeongnam', 'year': 2022, 'poll_gap':  -8.2, '민주당승': 0},  # 박완수 당선
    {'region': 'Gyeongbuk', 'year': 2022, 'poll_gap': -51.3, '민주당승': 0},  # 이철우 당선
    {'region': 'Jeju',      'year': 2022, 'poll_gap':   8.9, '민주당승': 1},  # 오영훈 당선
    # ── 7회 (2018)
    {'region': 'Seoul',     'year': 2018, 'poll_gap':  22.1, '민주당승': 1},  # 박원순 당선
    {'region': 'Busan',     'year': 2018, 'poll_gap':  14.3, '민주당승': 1},  # 오거돈 당선
    {'region': 'Daegu',     'year': 2018, 'poll_gap': -11.2, '민주당승': 0},  # 권영진 당선
    {'region': 'Incheon',   'year': 2018, 'poll_gap':  19.8, '민주당승': 1},  # 박남춘 당선
    {'region': 'Gyeonggi',  'year': 2018, 'poll_gap':  18.5, '민주당승': 1},  # 이재명 당선
    {'region': 'Gangwon',   'year': 2018, 'poll_gap':  26.3, '민주당승': 1},  # 최문순 당선
    {'region': 'Chungbuk',  'year': 2018, 'poll_gap':  28.1, '민주당승': 1},  # 이시종 당선
    {'region': 'Chungnam',  'year': 2018, 'poll_gap':  24.7, '민주당승': 1},  # 양승조 당선
    {'region': 'Jeonbuk',   'year': 2018, 'poll_gap':  48.2, '민주당승': 1},  # 송하진 당선
    {'region': 'Gyeongnam', 'year': 2018, 'poll_gap':   7.3, '민주당승': 1},  # 김경수 당선
    {'region': 'Gyeongbuk', 'year': 2018, 'poll_gap': -15.1, '민주당승': 0},  # 이철우 당선
    {'region': 'Jeju',      'year': 2018, 'poll_gap':   9.8, '민주당승': 1},  # 문대림 당선
]

df_hist = pd.DataFrame(historical_data)

# =============================================================================
# 2. 9회 경합 지역 (여론M 최신 여론조사 기준)
# =============================================================================
contest = pd.DataFrame([
    {'region': 'Seoul',     'cand1': '정원오', 'cand2': '오세훈', 'poll_gap':  3.9},
    {'region': 'Daegu',     'cand1': '김부겸', 'cand2': '추경호', 'poll_gap': 10.1},
    {'region': 'Jeonbuk',   'cand1': '김관영', 'cand2': '이원택', 'poll_gap':  3.0},
    {'region': 'Gyeongnam', 'cand1': '김경수', 'cand2': '박완수', 'poll_gap':  0.2},
])

# =============================================================================
# 3. 전처리
# =============================================================================
X = df_hist[['poll_gap']].values
y = df_hist['민주당승'].values

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# =============================================================================
# 4. 모델 정의
# =============================================================================
class TinyMLP(nn.Module):
    """
    샘플 수가 적으므로 (24개) 은닉층 최소화 + 강한 Dropout 정규화
    BCEWithLogitsLoss 사용 (수치 안정성)
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 8),
            nn.Tanh(),
            nn.Dropout(0.5),
            nn.Linear(8, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze()

def train_mlp(X_tr, y_tr, epochs=800, lr=0.01):
    m       = TinyMLP()
    opt     = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=0.05)
    loss_fn = nn.BCEWithLogitsLoss()
    Xt      = torch.FloatTensor(X_tr.copy())
    yt      = torch.FloatTensor(y_tr.copy())
    for _ in range(epochs):
        m.train()
        opt.zero_grad()
        loss_fn(m(Xt), yt).backward()
        opt.step()
    return m

def predict_proba_mlp(model, X_in):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X_in.copy()))
        return torch.sigmoid(logits).numpy()

# =============================================================================
# 5. LOO 교차검증
# =============================================================================
loo = LeaveOneOut()

# ── LR
preds_lr = []
for tr, te in loo.split(X_sc):
    m = LogisticRegression(C=1.0, random_state=42)
    m.fit(X_sc[tr], y[tr])
    preds_lr.append(m.predict(X_sc[te])[0])
acc_lr = accuracy_score(y, preds_lr)

# ── MLP
preds_mlp, probs_mlp_loo = [], []
for tr, te in loo.split(X_sc):
    m    = train_mlp(X_sc[tr], y[tr])
    prob = float(predict_proba_mlp(m, X_sc[te]))
    probs_mlp_loo.append(prob)
    preds_mlp.append(1 if prob > 0.5 else 0)
acc_mlp = accuracy_score(y, preds_mlp)

print(f'Logistic Regression LOO 정확도 : {acc_lr:.2%} ({int(acc_lr*len(y))}/{len(y)})')
print(f'MLP            LOO 정확도      : {acc_mlp:.2%} ({int(acc_mlp*len(y))}/{len(y)})')

# =============================================================================
# 6. Platt Scaling (MLP 확률 보정)
# =============================================================================
platt = LogisticRegression(C=1.0)
platt.fit(np.array(probs_mlp_loo).reshape(-1, 1), y)

# =============================================================================
# 7. 최종 모델 (전체 학습)
# =============================================================================
model_final = train_mlp(X_sc, y)
lr_final    = LogisticRegression(C=1.0, random_state=42)
lr_final.fit(X_sc, y)

# =============================================================================
# 8. 경합 지역 당락 예측
# =============================================================================
print('\n' + '=' * 60)
print('  2026 제9회 지방선거 경합지역 당락 예측')
print('  모델: TinyMLP + Platt Scaling 보정 + LR 앙상블')
print('=' * 60)

results = []
for _, row in contest.iterrows():
    x_raw = np.array([[row['poll_gap']]])
    x_sc  = scaler.transform(x_raw)

    # MLP raw 확률 → Platt 보정
    prob_raw  = float(predict_proba_mlp(model_final, x_sc))
    prob_mlp  = float(platt.predict_proba([[prob_raw]])[0][1])

    # LR 확률
    prob_lr   = float(lr_final.predict_proba(x_sc)[0][1])

    # 앙상블 (MLP 60% + LR 40%)
    prob_민주  = prob_mlp * 0.6 + prob_lr * 0.4
    prob_상대  = 1 - prob_민주

    winner     = row['cand1'] if prob_민주 > 0.5 else row['cand2']
    prob_win   = prob_민주    if prob_민주 > 0.5 else prob_상대
    margin     = abs(prob_민주 - 0.5)

    if margin < 0.10:
        confidence = '🔴 초경합'
    elif margin < 0.20:
        confidence = '🟡 경합'
    else:
        confidence = '🟢 우세'

    # 막대 시각화
    n1  = int(prob_민주 * 30)
    n2  = 30 - n1
    bar1 = '█' * n1 + '░' * n2
    bar2 = '█' * n2 + '░' * n1

    print(f"\n[{row['region']}]  여론조사 격차: {row['poll_gap']:+.1f}%p  {confidence}")
    print(f"  {row['cand1']:5s}  {bar1}  {prob_민주:.1%}")
    print(f"  {row['cand2']:5s}  {bar2}  {prob_상대:.1%}")
    print(f"  → 예측 당선: ✅ {winner}  ({prob_win:.1%})")

    results.append({
        'region'              : row['region'],
        'cand1'               : row['cand1'],
        'cand2'               : row['cand2'],
        'poll_gap'            : row['poll_gap'],
        f"{row['cand1']}_당선확률": round(prob_민주, 4),
        f"{row['cand2']}_당선확률": round(prob_상대, 4),
        '예측당선'             : winner,
        '신뢰도'               : confidence,
    })

print('\n' + '=' * 60)
print(f'  LR 정확도: {acc_lr:.2%}  |  MLP 정확도: {acc_mlp:.2%}')
print(f'  앙상블: MLP 60% + LR 40%')
print('=' * 60)

# =============================================================================
# 9. 결과 저장
# =============================================================================
file_path = r"C:\SKN_2ND_PROJECT_1TEAM\SKN31-2nd-1Team\data\processed"

df_result = pd.DataFrame(results)
df_result.to_csv(f"{file_path}/contest_prediction_9th.csv", index=False, encoding='utf-8-sig')
print(f"\n저장완료: contest_prediction_9th.csv")
print(df_result.to_string(index=False))

Logistic Regression LOO 정확도 : 91.67% (22/24)
MLP            LOO 정확도      : 91.67% (22/24)

  2026 제9회 지방선거 경합지역 당락 예측
  모델: TinyMLP + Platt Scaling 보정 + LR 앙상블

[Seoul]  여론조사 격차: +3.9%p  🔴 초경합
  정원오    ███████████████░░░░░░░░░░░░░░░  52.0%
  오세훈    ███████████████░░░░░░░░░░░░░░░  48.0%
  → 예측 당선: ✅ 정원오  (52.0%)

[Daegu]  여론조사 격차: +10.1%p  🟡 경합
  김부겸    ██████████████████░░░░░░░░░░░░  61.1%
  추경호    ████████████░░░░░░░░░░░░░░░░░░  38.9%
  → 예측 당선: ✅ 김부겸  (61.1%)

[Jeonbuk]  여론조사 격차: +3.0%p  🔴 초경합
  김관영    ███████████████░░░░░░░░░░░░░░░  50.6%
  이원택    ███████████████░░░░░░░░░░░░░░░  49.4%
  → 예측 당선: ✅ 김관영  (50.6%)

[Gyeongnam]  여론조사 격차: +0.2%p  🔴 초경합
  김경수    █████████████░░░░░░░░░░░░░░░░░  46.3%
  박완수    █████████████████░░░░░░░░░░░░░  53.7%
  → 예측 당선: ✅ 박완수  (53.7%)

  LR 정확도: 91.67%  |  MLP 정확도: 91.67%
  앙상블: MLP 60% + LR 40%

저장완료: contest_prediction_9th.csv
   region cand1 cand2  poll_gap  정원오_당선확률  오세훈_당선확률 예측당선   신뢰도  김부겸_당선확률  추경호_당선확률  김관영_당선확률  이원택_당선확률  김경수_당선확률  박완수_당선확률
   